# CARLA Python API - Project 5: Cooperative Roadside Assistant (V2X / I2V)

This notebook contains exactly 3 sections:
1. **Infrastructure Sensing & World Transformation** (Theory, camera setups, and 2D-to-3D projection layout)
2. **V2X MQTT Server Architecture & Message Serialization** (Live MQTT client setup, network delay emulation, and protocol payloads)
3. **End-to-End System Integration & Occlusion Scenarios** (The runnable project execution with full evaluation metrics)

## CARLA docs
- Main docs: https://carla.readthedocs.io/en/latest/
- Sensors reference: https://carla.readthedocs.io/en/latest/ref_sensors/
- Python API: https://carla.readthedocs.io/en/latest/python_api/

In [2]:
import carla
import time
import random
import cv2
import queue
import threading
import json
import math
import numpy as np
from datetime import datetime

# Initialize client and connect to the CARLA server daemon
client = carla.Client("localhost", 2000)
client.set_timeout(10.0)
world = client.get_world()
spectator = world.get_spectator()
blueprint_library = world.get_blueprint_library()

In [3]:
def move_spectator_to(transform, spectator, distance=12.0, z=6.0, pitch=-25.0):
    """Utility to orient the editor spectator view behind an active actor."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))

def safe_destroy(actors):
    """Safely removes lists of actors under teardown scenarios."""
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass

---
## 1. Infrastructure Sensing & World Transformation

### Mathematical Foundation: 2D Image Space to 3D World Space
To send meaningful target tracking coordinates to an Ego vehicle, a static roadside infrastructure camera must map a detected object's pixel coordinate $(u, v)$ back into a 3D World coordinate $(X_w, Y_w, Z_w)$.

This is achieved using the camera Intrinsic Matrix ($K$) and Extrinsic Matrix ($[R|t]$):

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

Given that the ground plane can be structurally approximated ($Z_w \approx 0$), we solve for the scaling factor $\lambda$ to project inverse coordinates from the pixel plane back through the inverted rigid transformation matrix of the camera mount location.

### Configurable Parameters Guide
- **Camera Extrinsics**: Positioning height ($z \ge 6.0\text{m}$) and dramatic downward pitch ($\text{pitch} \approx -35^\circ$) are vital to eliminate self-occlusion artifacts within blind intersections.
- **Intrinsic Matrix Fields**: Tuning Resolution ($W, H$) and Horizontal Field of View ($\text{FOV}$) determines pixel density per meter at long range.

In [4]:
def get_camera_intrinsic_matrix(width, height, fov):
    """Computes the intrinsic matrix K for a pinhole camera model."""
    focal = width / (2.0 * np.tan(fov * np.pi / 360.0))
    K = np.identity(3)
    K[0, 0] = focal
    K[1, 1] = focal
    K[0, 2] = width / 2.0
    K[1, 2] = height / 2.0
    return K

def spawn_roadside_camera(world, transform, width=800, height=600, fov=90, tick=0.05):
    """Spawns a static infrastructure sensor frame."""
    bp = world.get_blueprint_library().find("sensor.camera.rgb")
    bp.set_attribute("image_size_x", str(width))
    bp.set_attribute("image_size_y", str(height))
    bp.set_attribute("fov", str(fov))
    bp.set_attribute("sensor_tick", str(tick))
    return world.spawn_actor(bp, transform)

def image_to_bgr(image):
    """Converts CARLA raw image array to standard BGR layout."""
    arr = np.frombuffer(image.raw_data, dtype=np.uint8)
    arr = np.reshape(arr, (image.height, image.width, 4))
    return arr[:, :, :3].copy()

In [ ]:
# ==============================================================================
# CARLA NATIVE SENSOR AND TELEMETRY TEST RUNNER (FIXED SPAWN)
# ==============================================================================

print(">> Cleaning up any leftover vehicles in the world...")
# Automatically find and remove previously spawned vehicles to prevent collision errors
for actor in world.get_actors().filter("vehicle.*.*"):
    try:
        actor.destroy()
        print(f" Removed leftover vehicle ID: {actor.id}")
    except RuntimeError:
        pass

print("\n>> Initializing CARLA API Testing Loop...")

# 1. Access the CARLA World Settings for Synchronous Execution
original_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05  # Fixed time step increments of 50ms
world.apply_settings(settings)

# Track actors spawned natively in this cell
native_test_actors = []

try:
    # 2. Query Blueprint Library for Standard Targets
    tesla_bp = blueprint_library.filter("vehicle.tesla.model3")[0]
    ped_bp = blueprint_library.filter("walker.pedestrian.0001")[0]
    
    # 3. Native Spawning via world.spawn_actor()
    # Shifted x slightly to 118.0 and z to 2.0 to drop cleanly onto the road map surface
    ego_spawn_tf = carla.Transform(carla.Location(x=118.0, y=132.0, z=2.0), carla.Rotation(yaw=0.0))
    test_ego = world.spawn_actor(tesla_bp, ego_spawn_tf)
    native_test_actors.append(test_ego)
    
    # Pedestrian Spawn Location
    ped_spawn_tf = carla.Transform(carla.Location(x=152.0, y=142.0, z=1.5), carla.Rotation(yaw=-90.0))
    test_ped = world.spawn_actor(ped_bp, ped_spawn_tf)
    native_test_actors.append(test_ped)
    
    print(f" Successfully spawned Ego (ID: {test_ego.id}) and Pedestrian (ID: {test_ped.id})")
    
    # 4. Apply Initial Control Vectors
    test_ped.apply_control(carla.WalkerControl(direction=carla.Vector3D(0, -1, 0), speed=1.5))
    test_ego.apply_control(carla.VehicleControl(throttle=0.4, brake=0.0, steer=0.0))
    
    # Tick the world to instantiate entities
    world.tick()
    
    # 5. Core Simulation and Observation Loop (Runs for 100 simulator ticks)
    for frame in range(1000):
        # Frame tick signals the server simulator to advance one time step
        world.tick()
        
        # Pull native spatial Transform objects from runtime instances
        ego_transform = test_ego.get_transform()
        ped_transform = test_ped.get_transform()
        
        # Position the Spectator camera perspective procedurally behind the Ego Vehicle
        move_spectator_to(ego_transform, spectator, distance=15.0, z=6.0, pitch=-22.0)
        
        # Calculate straight-line spatial distance using math methods on carla.Location
        distance = ego_transform.location.distance(ped_transform.location)
        
        # Extract precise physical velocity vector using get_velocity()
        velocity_vec = test_ego.get_velocity()
        speed_kmh = 3.6 * math.sqrt(velocity_vec.x**2 + velocity_vec.y**2 + velocity_vec.z**2)
        
        # 6. Apply Native Debug Floating Text to the 3D Render Window
        world.debug.draw_string(
            ego_transform.location + carla.Location(z=2.5), 
            f"SPEED: {speed_kmh:.1f} KM/H", 
            life_time=0.06, 
            color=carla.Color(0, 255, 0)
        )
        
        world.debug.draw_string(
            ped_transform.location + carla.Location(z=2.0), 
            f"DISTANCE TO EGO: {distance:.1f}m", 
            life_time=0.06, 
            color=carla.Color(255, 255, 0)
        )
        
        # Proactive braking check based strictly on local distance measurements
        if distance < 15.0:
            test_ego.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0))
            world.debug.draw_string(
                ego_transform.location + carla.Location(z=3.2), 
                "⚠️ LOCAL BRAKING TRIGGERED", 
                life_time=0.06, 
                color=carla.Color(255, 0, 0)
            )

        if frame % 20 == 0:
            print(f"   [Tick {frame}] Speed={speed_kmh:.1f} km/h | Range={distance:.1f}m")

finally:
    print(">> Cleaning up native testing assets...")
    
    # Clean up actors spawned inside this specific execution block
    for actor in native_test_actors:
        if actor is not None and actor.is_alive:
            actor.destroy()
            
    # CRITICAL: Restore original simulator configurations 
    world.apply_settings(original_settings)
    print(">> Synchronous mode disabled. Environment returned to baseline state.")

>> Cleaning up any leftover vehicles in the world...

>> Initializing CARLA API Testing Loop...
 Successfully spawned Ego (ID: 351) and Pedestrian (ID: 352)
   [Tick 0] Speed=3.5 km/h | Range=35.4m
   [Tick 20] Speed=1.1 km/h | Range=35.3m
   [Tick 40] Speed=0.2 km/h | Range=34.8m
   [Tick 60] Speed=0.2 km/h | Range=34.5m
   [Tick 80] Speed=0.2 km/h | Range=34.2m
   [Tick 100] Speed=0.2 km/h | Range=34.0m
   [Tick 120] Speed=0.2 km/h | Range=33.9m
   [Tick 140] Speed=0.2 km/h | Range=33.9m
   [Tick 160] Speed=0.2 km/h | Range=33.9m
   [Tick 180] Speed=0.2 km/h | Range=33.9m
   [Tick 200] Speed=0.2 km/h | Range=34.1m
   [Tick 220] Speed=0.2 km/h | Range=34.0m
   [Tick 240] Speed=0.2 km/h | Range=34.0m
   [Tick 260] Speed=0.2 km/h | Range=34.0m
   [Tick 280] Speed=0.2 km/h | Range=34.0m
   [Tick 300] Speed=0.2 km/h | Range=34.0m
   [Tick 320] Speed=0.2 km/h | Range=33.9m
   [Tick 340] Speed=0.1 km/h | Range=33.9m
   [Tick 360] Speed=0.1 km/h | Range=33.9m
   [Tick 380] Speed=0.1 km/h | R